# Part E — Determinants

**Final Project: The Linear Algebra Behind AI — A Regression Problem**

Before you can solve for regression coefficients using matrix inversion, you need to know the inversion is actually possible.

---
**Student name:** _______________________   **Date:** _______________________

---


**What to do:** Compute det(XᵀX) on your training data (after your Part D transformation) before attempting to invert it.

In [1]:
# Run this first: recall variables saved from the previous part's notebook.
import os, numpy as np
try:
    from google.colab import drive
    drive.mount('/content/drive')
    data = np.load('/content/drive/MyDrive/partD_data.npz', allow_pickle=True)
    X_train, X_test = data['X_train'], data['X_test']
    y_train, y_test = data['y_train'], data['y_test']
except Exception:
    if os.path.exists('partD_data.npz'):
        data = np.load('partD_data.npz', allow_pickle=True)
        X_train, X_test = data['X_train'], data['X_test']
        y_train, y_test = data['y_train'], data['y_test']
    else:
        get_ipython().run_line_magic('store', '-r X_train X_test y_train y_test')
print('Loaded X_train:', X_train.shape, ' X_test:', X_test.shape)

Loaded X_train: (5285, 4)  X_test: (1322, 4)


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

In [3]:
# Compute X_train^T X_train and its determinant
XtX_train = X_train.T @ X_train
det_XtX_train = np.linalg.det(XtX_train)
print("X_train^T X_train:\n", XtX_train)
print(f"det(X_train^T X_train) = {det_XtX_train:.4f}")

X_train^T X_train:
 [[5286.8236  -58.9777 -126.1221  -63.5044]
 [ -58.9777 5307.7899   84.0525   55.9671]
 [-126.1221   84.0525 5293.5145  -81.2648]
 [ -63.5044   55.9671  -81.2648 5298.5532]]
det(X_train^T X_train) = 785932759224407.3750


In [4]:
# Decide, based on the determinant, whether XtX_train is invertible.
# Also compute its condition number for a stability check.
cond_number = np.linalg.cond(XtX_train)
print(f"Condition number = {cond_number:.4f}")
if not np.isclose(det_XtX_train, 0.0):
    print("Conclusion: det(X_train^T X_train) != 0, so X_train^T X_train is invertible.")
else:
    print("Conclusion: det(X_train^T X_train) is close to 0; matrix is singular or nearly singular.")

Condition number = 1.0747
Conclusion: det(X_train^T X_train) != 0, so X_train^T X_train is invertible.


### Answer these in your report

**Q17.** What is det(XᵀX) for your training data?

> *Your answer:* For our standardized training matrix $X_{\text{train}} \in \mathbb{R}^{5285 \times 4}$ (80% of the 6,607 students), $\det(X_{\text{train}}^T X_{\text{train}}) = 785,932,759,224,407.3750$ (approximately $7.8593 \times 10^{14}$). Because each standardized feature has unit variance, the diagonal entries of $X_{\text{train}}^T X_{\text{train}}$ are close to $n_{\text{train}} = 5,285$ ($[5286.8236, 5307.7899, 5293.5145, 5298.5532]$) and the off-diagonal entries are relatively small, making the determinant close to $(5,285)^4$ and far from zero.

**Q18.** Based on that value, is XᵀX invertible? If it were numerically singular, what would you do instead?

> *Your answer:* Yes, because $\det(X_{\text{train}}^T X_{\text{train}}) = 7.8593 \times 10^{14} \neq 0$, $X_{\text{train}}^T X_{\text{train}}$ is nonsingular and invertible, so the normal equations $\hat{\beta} = (X_{\text{train}}^T X_{\text{train}})^{-1} X_{\text{train}}^T y_{\text{train}}$ have a unique closed-form solution. If $X_{\text{train}}^T X_{\text{train}}$ were numerically singular ($\det \approx 0$), we could instead: (1) drop or combine the redundant/collinear features (or use PCA to keep only directions with nonzero variance), (2) compute the least-squares solution using the Moore–Penrose pseudoinverse (`np.linalg.pinv`) or SVD-based solver (`np.linalg.lstsq`), or (3) use Ridge regression ($L_2$ regularization), replacing $X^T X$ with $X^T X + \lambda I$ ($\lambda > 0$) so all eigenvalues are shifted above zero and the matrix becomes invertible.

**Q19.** Describe a realistic situation in your own dataset (e.g., two nearly duplicate features) that could push this determinant close to zero.

> *Your answer:* In our student dataset, $\det(X_{\text{train}}^T X_{\text{train}})$ would be pushed close to zero if we included two features that measure nearly the same quantity—for example, keeping `Hours_Studied` (weekly study hours) while also adding a column `Daily_Study_Hours` $\approx \text{Hours\_Studied} / 7$ (with slight rounding noise), or including both `Attendance` (percentage of classes attended) and `Classes_Missed` out of a fixed semester schedule. Those two columns would be nearly collinear, causing the smallest eigenvalue of $X_{\text{train}}^T X_{\text{train}}$—and thus the determinant, which equals the product of the eigenvalues—to drop close to zero.

**Q20.** Look up or compute the condition number of XᵀX. How does a large condition number relate to the determinant, and what does it suggest about the stability of your model?

> *Your answer:* The condition number of $X_{\text{train}}^T X_{\text{train}}$ is $\kappa(X_{\text{train}}^T X_{\text{train}}) = \frac{\lambda_{\max}}{\lambda_{\min}} = 1.0747$, the ratio of its largest to smallest eigenvalue. Because our four standardized features are nearly orthogonal, all four eigenvalues are close to $n_{\text{train}} = 5,285$, giving a condition number of $1.0747$ (very close to the ideal value of $1$), which indicates an exceptionally well-conditioned and numerically stable model. In general, since $\det(X^T X) = \prod_{i=1}^p \lambda_i$, near-collinearity drives the smallest eigenvalue $\lambda_{\min} \to 0$, which simultaneously pushes $\det(X^T X) \to 0$ (for normalized columns) and causes $\kappa(X^T X) = \lambda_{\max}/\lambda_{\min} \to \infty$. A large condition number means the matrix is close to singular and the regression model is numerically unstable: small changes or noise in the data can cause large swings in the estimated coefficients $\hat{\beta}$.

### Save your work for the next notebook

In [5]:
%store XtX_train
np.savez('partE_data.npz', XtX_train=XtX_train, det_XtX_train=det_XtX_train, cond_number=cond_number)
try:
    from google.colab import drive
    if os.path.exists('/content/drive/MyDrive'):
        np.savez('/content/drive/MyDrive/partE_data.npz', XtX_train=XtX_train, det_XtX_train=det_XtX_train, cond_number=cond_number)
except Exception:
    pass
print("Saved XtX_train for Part F.")

Saved XtX_train for Part F.
